In [ ]:

import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# %%
# Nos quedamos solo con energía, no agua
df_train = df_energia_diff.copy()

# Si quieres, evita negativos raros tras diff
for col in cols_pisos_energia:
    df_train[col] = df_train[col].clip(lower=0)

# %%
# Pasar a formato largo: una fila = una hora de una vivienda
df_long = df_train.melt(
    id_vars="fecha",
    value_vars=cols_pisos_energia,
    var_name="vivienda",
    value_name="consumo"
)

df_long = df_long.dropna().copy()
df_long = df_long.sort_values(["vivienda", "fecha"]).reset_index(drop=True)

df_long.head()

# %%
# Variables temporales
df_long["hour"] = df_long["fecha"].dt.hour
df_long["dayofweek"] = df_long["fecha"].dt.dayofweek
df_long["day"] = df_long["fecha"].dt.day
df_long["month"] = df_long["fecha"].dt.month
df_long["is_weekend"] = (df_long["dayofweek"] >= 5).astype(int)

# %%
# Lags por vivienda
df_long["lag_1"] = df_long.groupby("vivienda")["consumo"].shift(1)
df_long["lag_24"] = df_long.groupby("vivienda")["consumo"].shift(24)
df_long["lag_168"] = df_long.groupby("vivienda")["consumo"].shift(168)

# %%
# Medias móviles por vivienda
df_long["rolling_24"] = (
    df_long.groupby("vivienda")["consumo"]
    .transform(lambda s: s.shift(1).rolling(24).mean())
)

df_long["rolling_168"] = (
    df_long.groupby("vivienda")["consumo"]
    .transform(lambda s: s.shift(1).rolling(168).mean())
)

# %%
# Codificar vivienda como número
le = LabelEncoder()
df_long["vivienda_id"] = le.fit_transform(df_long["vivienda"])

# %%
# Eliminar filas con nulos generados por lags/rolling
df_model = df_long.dropna().copy()

print("Shape final del dataset de entrenamiento:", df_model.shape)
df_model.head()

# %%
# División temporal: 70% train, 15% valid, 15% test
fechas_ordenadas = np.array(sorted(df_model["fecha"].unique()))
n = len(fechas_ordenadas)

train_end = fechas_ordenadas[int(n * 0.70)]
valid_end = fechas_ordenadas[int(n * 0.85)]

train = df_model[df_model["fecha"] <= train_end].copy()
valid = df_model[(df_model["fecha"] > train_end) & (df_model["fecha"] <= valid_end)].copy()
test = df_model[df_model["fecha"] > valid_end].copy()

print("Train:", train.shape)
print("Valid:", valid.shape)
print("Test:", test.shape)

# %%
features = [
    "vivienda_id",
    "hour",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_24",
    "rolling_168"
]

target = "consumo"

X_train = train[features]
y_train = train[target]

X_valid = valid[features]
y_valid = valid[target]

X_test = test[features]
y_test = test[target]

# %%
model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# %%
# Predicción en test
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

# %%
# Guardar resultados
results = test[["fecha", "vivienda", "consumo"]].copy()
results["pred"] = y_pred

results.head()

# %%
# Visualizar una vivienda concreta
vivienda_ejemplo = "energia1C"   # cambia esto por la que quieras

r = results[results["vivienda"] == vivienda_ejemplo].copy()

plt.figure(figsize=(14,5))
plt.plot(r["fecha"], r["consumo"], label="Real")
plt.plot(r["fecha"], r["pred"], label="Predicho")
plt.title(f"Consumo real vs predicho - {vivienda_ejemplo}")
plt.xlabel("Fecha")
plt.ylabel("Consumo horario")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# %%
# Importancia de variables
importancias = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,5))
importancias.plot(kind="bar")
plt.title("Importancia de variables")
plt.ylabel("Importancia")
plt.tight_layout()
plt.show()